# Proyecto Big Data: Batch Processing Pipeline
**Autor:** Bryan Edgardo Romo Gonzalez

In [1]:
from spark_utils import SparkUtils
from pyspark.sql.functions import col, round, when, current_timestamp, lit, sum, count

su = SparkUtils("Proyecto_BigData_BryanRomo", "spark://spark-master:7077")
su.spark


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/25 03:36:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
!ls /opt/spark/work-dir/data/proyecto

ecommerce_batch_1.csv  ecommerce_test.csv


## 1. Definicion de Esquema

In [3]:
columns_info = [
    ("transaction_id", "string"),
    ("user_id", "int"),
    ("timestamp", "string"),
    ("product_id", "string"),
    ("category", "string"),
    ("price", "double"),
    ("quantity", "int"),
    ("region", "string"),
    ("order_status", "string")
]

custom_schema = SparkUtils.generate_schema(columns_info)
print("Esquema cargado correctamente.")

Esquema cargado correctamente.


## 2. Ingesta de Datos
Carga de 5 archivos para prueba

In [4]:
# Ruta a la carpeta que contiene el archivos CSV
input_path = "/opt/spark/work-dir/data/proyecto/ecommerce_test.csv"

df_raw = su.spark.read.option("header", "true").schema(custom_schema).csv(input_path)

df_raw.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- order_status: string (nullable = true)



In [5]:
df_raw.show(5)

+--------------------+-------+-------------------+----------+-------------+-------+--------+-------------+------------+
|      transaction_id|user_id|          timestamp|product_id|     category|  price|quantity|       region|order_status|
+--------------------+-------+-------------------+----------+-------------+-------+--------+-------------+------------+
|0494f9ea-7730-4ad...| 589137|2025-05-10T08:46:00|  PRD-8129|       Health| 193.92|       6|North America|   Cancelled|
|f5f1b5fe-0470-461...| 278155|2025-09-07T07:23:00|  PRD-9333|     Clothing|   NULL|       3|North America|    Refunded|
|a133dba6-6062-441...| 398605|2025-05-22T15:11:00|  PRD-3862|        Books| 850.85|       3|         Asia|   Completed|
|0680f8b2-3245-426...| 766875|2025-08-08T13:48:00|  PRD-9091|   Automotive| 549.36|       4|       Africa|   Completed|
|bf3496cb-c25b-43d...| 405206|2025-10-23T18:49:00|  PRD-3925|Home & Garden|1093.01|       8|South America|   Completed|
+--------------------+-------+----------

## Perfilado de Datos (Sanity Checks)
Antes de aplicar transformaciones, verificamos el dataset

In [7]:
print("--- AUDITORÍA DE CALIDAD DE DATOS ---")

# 1. Búsqueda de Duplicados
total_rows = df_raw.count()
unique_ids = df_raw.select("transaction_id").distinct().count()
duplicates = total_rows - unique_ids

# 2. Búsqueda de Nulos
null_users = df_raw.filter(col("user_id").isNull()).count()
null_prices = df_raw.filter(col("price").isNull()).count()

print(f"Total de registros evaluados: {total_rows}")
print(f"-> Duplicados detectados: {duplicates}")
print(f"-> Usuarios nulos detectados: {null_users}")
print(f"-> Precios nulos detectados: {null_prices}")

--- AUDITORÍA DE CALIDAD DE DATOS ---


[Stage 28:=============================>                            (1 + 1) / 2]

Total de registros evaluados: 960000
-> Duplicados detectados: 47870
-> Usuarios nulos detectados: 47949
-> Precios nulos detectados: 47964


## 2. Transformación 1: Data Cleaning
* **Duplicados:** Se eliminan transacciones repetidas
* **Registros Malformados/Nulos:** Se descartan filas sin identificador de cliente y se cambian los precios vacíos

In [8]:
df_clean = df_raw.dropDuplicates(["transaction_id"]) \
                 .dropna(subset=["user_id"]) \
                 .fillna({"price": 0.0})

print("Limpieza de duplicados y nulos completada.")

df_clean.show(5)

Limpieza de duplicados y nulos completada.


[Stage 31:=============================>                            (1 + 1) / 2]

+--------------------+-------+-------------------+----------+-------------+-------+--------+-------------+------------+
|      transaction_id|user_id|          timestamp|product_id|     category|  price|quantity|       region|order_status|
+--------------------+-------+-------------------+----------+-------------+-------+--------+-------------+------------+
|00001d48-c634-4da...| 232954|2025-09-04T15:39:00|  PRD-2750|       Sports| 751.21|       7|North America|   Cancelled|
|0000e764-a2dc-432...| 881099|2025-08-05T19:13:00|  PRD-6856|Home & Garden|1220.99|       5|      Oceania|   Cancelled|
|0001030e-fe46-45a...| 867008|2025-06-24T03:06:00|  PRD-4400|       Sports|1491.55|       3|North America|     Pending|
|000144d0-3896-4e8...| 330967|2025-07-05T06:11:00|  PRD-8178|         Toys| 758.11|       6|      Oceania|   Completed|
|00019c76-bb70-4a0...| 675627|2025-05-01T06:13:00|  PRD-7329|  Electronics| 501.94|      10|       Europe|   Completed|
+--------------------+-------+----------

In [9]:
print("--- PRUEBA RÁPIDA DE LIMPIEZA ---")

# 1. Demostrar que se eliminó basura
print(f"1. Filas originales: {df_raw.count()}")
print(f"2. Filas limpias: {df_clean.count()}")

# 2. Demostrar que el dropna() y dropDuplicates() funcionaron
nulos = df_clean.filter(col("user_id").isNull()).count()
duplicados = df_clean.count() - df_clean.select("transaction_id").distinct().count()

print(f"\n3. Errores restantes:")
print(f"   - Nulos: {nulos}")
print(f"   - Duplicados: {duplicados}")

--- PRUEBA RÁPIDA DE LIMPIEZA ---


1. Filas originales: 960000


2. Filas limpias: 866511


[Stage 55:=============================>                            (1 + 1) / 2]


3. Errores restantes:
   - Nulos: 0
   - Duplicados: 0


## 3. Transformación 2: Joins

In [10]:
# Catálogo de impuestos por categoría de producto
tax_data = [
    ("Electronics", 0.16), ("Clothing", 0.08), ("Home & Garden", 0.16), 
    ("Books", 0.00), ("Health", 0.00), ("Automotive", 0.16), 
    ("Toys", 0.16), ("Sports", 0.16)
]
df_taxes = su.spark.createDataFrame(tax_data, ["category", "tax_rate"])

# Left Join sobre la columna 'category'
df_joined = df_clean.join(df_taxes, on="category", how="left")

print("Catálogo de impuestos integrado exitosamente.")
df_joined.show(5)

Catálogo de impuestos integrado exitosamente.


+-------------+--------------------+-------+-------------------+----------+-------+--------+-------------+------------+--------+
|     category|      transaction_id|user_id|          timestamp|product_id|  price|quantity|       region|order_status|tax_rate|
+-------------+--------------------+-------+-------------------+----------+-------+--------+-------------+------------+--------+
|       Sports|00001d48-c634-4da...| 232954|2025-09-04T15:39:00|  PRD-2750| 751.21|       7|North America|   Cancelled|    0.16|
|Home & Garden|0000e764-a2dc-432...| 881099|2025-08-05T19:13:00|  PRD-6856|1220.99|       5|      Oceania|   Cancelled|    0.16|
|       Sports|0001030e-fe46-45a...| 867008|2025-06-24T03:06:00|  PRD-4400|1491.55|       3|North America|     Pending|    0.16|
|         Toys|000144d0-3896-4e8...| 330967|2025-07-05T06:11:00|  PRD-8178| 758.11|       6|      Oceania|   Completed|    0.16|
|  Electronics|00019c76-bb70-4a0...| 675627|2025-05-01T06:13:00|  PRD-7329| 501.94|      10|     

## 4. Transformación 3: Column Derivation

In [11]:
# Derivación de subtotales, monto de impuesto y total final
df_derived = df_joined.withColumn("subtotal", round(col("price") * col("quantity"), 2)) \
                      .withColumn("tax_amount", round(col("subtotal") * col("tax_rate"), 2)) \
                      .withColumn("total_sale_with_tax", round(col("subtotal") + col("tax_amount"), 2))

print("Nuevas columnas financieras calculadas.")
df_derived.show(5)

Nuevas columnas financieras calculadas.


+-------------+--------------------+-------+-------------------+----------+-------+--------+-------------+------------+--------+--------+----------+-------------------+
|     category|      transaction_id|user_id|          timestamp|product_id|  price|quantity|       region|order_status|tax_rate|subtotal|tax_amount|total_sale_with_tax|
+-------------+--------------------+-------+-------------------+----------+-------+--------+-------------+------------+--------+--------+----------+-------------------+
|       Sports|00001d48-c634-4da...| 232954|2025-09-04T15:39:00|  PRD-2750| 751.21|       7|North America|   Cancelled|    0.16| 5258.47|    841.36|            6099.83|
|Home & Garden|0000e764-a2dc-432...| 881099|2025-08-05T19:13:00|  PRD-6856|1220.99|       5|      Oceania|   Cancelled|    0.16| 6104.95|    976.79|            7081.74|
|       Sports|0001030e-fe46-45a...| 867008|2025-06-24T03:06:00|  PRD-4400|1491.55|       3|North America|     Pending|    0.16| 4474.65|    715.94|       

## 5. Transformación 4: Filtering and Sorting

In [12]:
df_filtered = df_derived.filter(col("order_status") == "Completed") \
                        .orderBy(col("timestamp").desc())

print("Datos filtrados por estatus y ordenados cronológicamente.")
df_filtered.show(5)

Datos filtrados por estatus y ordenados cronológicamente.


+-------------+--------------------+-------+-------------------+----------+------+--------+-------------+------------+--------+--------+----------+-------------------+
|     category|      transaction_id|user_id|          timestamp|product_id| price|quantity|       region|order_status|tax_rate|subtotal|tax_amount|total_sale_with_tax|
+-------------+--------------------+-------+-------------------+----------+------+--------+-------------+------------+--------+--------+----------+-------------------+
|       Sports|7ac1ba15-795b-449...| 639984|2026-01-01T00:00:00|  PRD-8568|954.53|       4|South America|   Completed|    0.16| 3818.12|     610.9|            4429.02|
|       Sports|eaee8d0b-cf62-442...| 359619|2025-12-31T23:59:00|  PRD-3287|490.26|       3|       Africa|   Completed|    0.16| 1470.78|    235.32|             1706.1|
|   Automotive|3daec316-5ac4-491...| 782736|2025-12-31T23:58:00|  PRD-2346|853.99|       9|North America|   Completed|    0.16| 7685.91|   1229.75|            8

## 6. Transformación 5: Aggregations

In [13]:
df_metrics = df_filtered.groupBy("region") \
                        .agg(
                            sum("total_sale_with_tax").alias("total_regional_revenue"),
                            count("transaction_id").alias("total_successful_orders")
                        )

print("Reporte de Rendimiento por Región:")
df_metrics.show()

Reporte de Rendimiento por Región:


[Stage 86:======================================>                   (2 + 1) / 3]

+-------------+----------------------+-----------------------+
|       region|total_regional_revenue|total_successful_orders|
+-------------+----------------------+-----------------------+
|       Africa|   3.128838317099994E8|                  71527|
|         Asia|   3.139292833600005E8|                  72333|
|South America|   3.168386995899995E8|                  72448|
|North America|   3.168474641800005E8|                  72661|
|       Europe|    3.16391546450001E8|                  72550|
|      Oceania|   3.139695436299997E8|                  71987|
+-------------+----------------------+-----------------------+



## 7. Persistencia de Datos Curados

In [14]:
output_path = "/opt/spark/work-dir/data/processed_output/"

df_filtered.write \
    .mode("overwrite") \
    .partitionBy("region") \
    .parquet(output_path)

print(f"Datos particionados por región y guardados en: {output_path}")

Datos particionados por región y guardados en: /opt/spark/work-dir/data/processed_output/


In [16]:
su.spark.stop()